### 1. Install dependencies

Installs the libraries this notebook needs: LangChain (for chaining retrieval and generation steps), a Hugging Face embedding model and local LLM, Chroma as the vector store, `rank_bm25` for keyword search, `pypdf` for reading PDFs, and MLflow for packaging and registering the final model.

In [0]:
# langchain / langchain-community / langchain-text-splitters: chain together retrieval + LLM steps
# langchain-huggingface: wrappers for local Hugging Face embedding models and text-generation pipelines
# langchain-chroma: Chroma vector store integration
# sentence-transformers: backing library for the embedding model
# chromadb: the vector database itself
# pypdf: PDF parsing, used by PyPDFLoader
# rank_bm25: keyword-based (BM25) search, used alongside vector search for retrieval
# mlflow: packages, registers, and serves the final model
%pip install langchain langchain-community langchain-huggingface langchain-chroma langchain-text-splitters sentence-transformers chromadb pypdf rank_bm25 mlflow -q

# Restart Python so the freshly installed packages are picked up in this session
dbutils.library.restartPython()


### 2. Load PDFs and split them into section-aware chunks

Every PDF in `PDF_FOLDER_PATH` is loaded page by page. Instead of cutting the text into arbitrary fixed-size blocks, the text is split on detected section headings — things like `SECTION 1:`, `3.2 Leave Policy`, `ARTICLE 4`, or an ALL-CAPS heading line — so each chunk stays a complete, coherent section. Any section that's still too long is further split with a standard recursive character splitter. Every chunk keeps track of which file, page, and section it came from; that metadata is what powers citations later on. The resulting chunks are embedded and stored in a Chroma vector database, and also pickled separately so a BM25 keyword index can be rebuilt from them later.

In [0]:
import glob
import os
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

PDF_FOLDER_PATH = "/Volumes/workspace/default/company_policy"
VECTOR_DB_PATH = "/tmp/chroma_policy_db"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Maximum size of a chunk before it gets sub-split further, plus the overlap
# used when sub-splitting (chars)
MAX_CHUNK_CHARS = 1500
CHUNK_OVERLAP = 150

# Regex that matches common policy/contract heading styles:
#   "SECTION 1: Travel Policy"   "Article 3 - Leave"   "4.2 Reimbursement"   "ALL CAPS HEADING"
HEADER_PATTERN = re.compile(
    r"(?m)^(?:"
    r"(?:SECTION|Section|ARTICLE|Article)\s+\d+[:.\-]?.*|"
    r"\d+(?:\.\d+)*\s+[A-Z][^\n]{0,80}|"
    r"[A-Z][A-Z0-9 &/,\-]{4,80}$"
    r")"
)

# Fallback splitter used when no headings are found in a document, and for
# sub-splitting any single section that's still too long on its own
_sub_splitter = RecursiveCharacterTextSplitter(chunk_size=MAX_CHUNK_CHARS, chunk_overlap=CHUNK_OVERLAP)


def split_by_sections(full_text):
    """Split document text on detected section headings, so each chunk is a
    complete section rather than an arbitrary slice of text.

    - If no headings are found anywhere in the document, falls back to a
      plain recursive character split over the whole text.
    - Any text appearing before the first heading (e.g. a title page or short
      intro) is kept as its own chunk with no section title.
    - Any single section that's still longer than MAX_CHUNK_CHARS is
      sub-split further.
    """
    matches = list(HEADER_PATTERN.finditer(full_text))

    if not matches:
        return [{"text": t, "section": None} for t in _sub_splitter.split_text(full_text)]

    pieces = []

    # Text before the first heading (title pages, short intros) gets its own chunk
    preamble = full_text[:matches[0].start()].strip()
    if preamble:
        if len(preamble) <= MAX_CHUNK_CHARS:
            pieces.append({"text": preamble, "section": None})
        else:
            for sub in _sub_splitter.split_text(preamble):
                pieces.append({"text": sub, "section": None})

    # Walk through each heading match and take everything up to the next heading
    # (or the end of the document) as that section's text
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        section_title = m.group().strip()
        section_text = full_text[start:end].strip()
        if not section_text:
            continue
        if len(section_text) <= MAX_CHUNK_CHARS:
            pieces.append({"text": section_text, "section": section_title})
        else:
            for sub in _sub_splitter.split_text(section_text):
                pieces.append({"text": sub, "section": section_title})
    return pieces


# 1. Find every PDF in the folder and load it page by page
pdf_files = glob.glob(f"{PDF_FOLDER_PATH}/*.pdf")
print(f"Found {len(pdf_files)} PDF files.")

doc_chunks = []
for file_path in pdf_files:
    print(f"Loading {file_path}...")
    pages = PyPDFLoader(file_path).load()

    # Concatenate all pages into one string, inserting an invisible page marker
    # before each page's text so the original page number can be recovered
    # after splitting on sections
    combined = ""
    for p in pages:
        page_num = p.metadata.get("page", 0) + 1  # PyPDFLoader pages are 0-indexed
        combined += f"\n\n<<PAGE:{page_num}>>\n\n{p.page_content}"

    chunks_before = len(doc_chunks)  # how many chunks existed before this file

    # Split the combined text into section-aware chunks, then recover the page
    # number for each chunk from its page marker and strip the marker out
    for piece in split_by_sections(combined):
        text = piece["text"]
        page_matches = re.findall(r"<<PAGE:(\d+)>>", text)
        page_num = int(page_matches[0]) if page_matches else None
        clean_text = re.sub(r"<<PAGE:\d+>>", "", text).strip()
        if not clean_text:
            continue
        doc_chunks.append(Document(
            page_content=clean_text,
            metadata={
                "source": file_path,
                "page": page_num,
                "section": piece["section"],
            }
        ))

    # If a PDF adds zero chunks, it likely has no readable text at all --
    # for example a scanned page saved as an image, which PyPDFLoader cannot
    # read without OCR. Flag it here instead of staying silent about it.
    chunks_added = len(doc_chunks) - chunks_before
    if chunks_added == 0:
        print(f"  WARNING: no text could be read from {file_path}. "
              f"It may be a scanned PDF with no selectable text, or an empty file.")
    else:
        print(f"  Added {chunks_added} chunks from this file.")

print(f"Created {len(doc_chunks)} section-aware chunks.")

# 2. Embed every chunk and persist the embeddings to a Chroma vector database on disk
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, model_kwargs={"device": "cpu"})

vector_store = Chroma.from_documents(
    documents=doc_chunks,
    embedding=embeddings,
    persist_directory=VECTOR_DB_PATH
)
print("Vector DB built and saved.")

# 3. Pickle the raw chunks separately too. BM25 isn't a persisted index like Chroma --
#    it's rebuilt in memory from the source documents each time it's needed, so the
#    served model needs access to these same chunks to rebuild it at serving time.
import pickle
CHUNKS_PATH = "/tmp/policy_doc_chunks.pkl"
with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(doc_chunks, f)
print(f"Chunks pickled to {CHUNKS_PATH} for BM25 rebuild at serving time.")


### 3. Shared RAG logic (`rag_core.py`)

Writes a small module, `rag_core.py`, containing the retrieval, prompting, and citation logic: merging keyword (BM25) and vector search results, formatting retrieved chunks into a prompt with citation tags, building a structured list of sources, and assembling the full retrieval-augmented chain. Both the notebook (for interactive testing, next cell) and the model file used for serving (`agent.py`, a few cells down) import this same module, so there's a single implementation of the RAG logic used everywhere.

In [0]:
rag_core_code = '''
"""
Shared retrieval-augmented generation (RAG) logic.

Used both by the notebook (for interactive testing) and by agent.py, the
model file that gets packaged and served through MLflow. Keeping this logic
in one module means both places always run the exact same code path.
"""

import os
from langchain_core.runnables import RunnableLambda, RunnableParallel
from langchain_core.output_parsers import StrOutputParser


SYSTEM_PROMPT = (
    "You are a helpful company policy assistant. Answer the user\'s question using "
    "ONLY the provided context. Every factual claim you make must be immediately "
    "followed by a citation in the form (Source: <file>, page <n>). "
    "If the context does not contain the answer, say you do not know -- do not guess."
)


def format_docs(docs):
    """Turn a list of retrieved chunks into the context string the LLM sees,
    with a citation tag (source file, page, section) in front of each chunk."""
    blocks = []
    for d in docs:
        src = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page", "?")
        section = d.metadata.get("section") or "General"
        blocks.append(f"[Source: {src}, page {page}, section: {section}]\\n{d.page_content}")
    return "\\n\\n---\\n\\n".join(blocks)


def get_sources(docs):
    """Build a structured, de-duplicated citation list (source file, page, section)
    from the retrieved chunks, so the caller can display exactly what was used
    to answer the question without relying on the model to cite itself correctly."""
    seen = set()
    sources = []
    for d in docs:
        src = os.path.basename(d.metadata.get("source", "unknown"))
        page = d.metadata.get("page")
        section = d.metadata.get("section") or "General"
        key = (src, page, section)
        if key in seen:
            continue
        seen.add(key)
        sources.append({"source": src, "page": page, "section": section})
    return sources


def make_hybrid_retriever(vector_retriever, bm25_retriever):
    """Combine keyword (BM25) and vector search results for a query.

    Both retrievers are queried, results are interleaved, and duplicate chunks
    (identical text content) are dropped. This is done with a small manual
    merge rather than LangChain\'s built-in EnsembleRetriever class, to keep
    this module independent of that class\'s exact import path across
    LangChain versions."""
    def hybrid_retrieve(query):
        vector_docs = vector_retriever.invoke(query)
        keyword_docs = bm25_retriever.invoke(query)
        seen = set()
        merged = []
        for pair in zip(keyword_docs, vector_docs):
            for doc in pair:
                if doc.page_content in seen:
                    continue
                seen.add(doc.page_content)
                merged.append(doc)
        return merged
    return hybrid_retrieve


def make_build_prompt(tokenizer):
    """Return a function that turns {context, question} into a fully formatted
    prompt string, using the given tokenizer\'s chat template."""
    def build_prompt(inputs):
        user_content = f"Context:\\n{inputs[\'context\']}\\n\\nQuestion: {inputs[\'question\']}"
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return build_prompt


def build_rag_chain(vector_retriever, bm25_retriever, tokenizer, llm):
    """Assemble the full RAG chain: given a question, retrieve chunks once
    using hybrid search, then build both the generated answer and the
    structured citation list from that same set of retrieved chunks."""
    hybrid_retrieve = make_hybrid_retriever(vector_retriever, bm25_retriever)
    build_prompt = make_build_prompt(tokenizer)

    def retrieve_and_package(query):
        docs = hybrid_retrieve(query)
        return {
            "context": format_docs(docs),
            "sources": get_sources(docs),
            "question": query,
        }

    retrieval_step = RunnableLambda(retrieve_and_package)

    rag_chain = (
        retrieval_step
        | RunnableParallel(
            answer=RunnableLambda(build_prompt) | llm | StrOutputParser(),
            sources=RunnableLambda(lambda x: x["sources"]),
        )
    )
    return rag_chain
'''

# Write the module to disk so it can be imported by name (both from this
# notebook and, later, from the packaged agent.py model)
with open("rag_core.py", "w") as f:
    f.write(rag_core_code)

print("rag_core.py written.")


### 4. Build the hybrid retriever, LLM, and RAG chain

Sets up two retrievers over the chunks created earlier — a vector (semantic) retriever and a BM25 (keyword) retriever — and loads a small local LLM to generate answers. `rag_core.build_rag_chain(...)` combines these into a single chain that retrieves relevant chunks for a question and returns both a generated answer and the list of sources it was based on.

In [0]:
import sys, os
sys.path.insert(0, os.getcwd())  # make rag_core.py (written above) importable

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_community.retrievers import BM25Retriever
from rag_core import build_rag_chain

# Small local instruction-tuned model used to generate answers.
# Swap this for a larger model (e.g. "Qwen/Qwen2.5-7B-Instruct") if GPU compute is available.
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
TOP_K = 6              # number of chunks each retriever (vector and BM25) returns
MAX_NEW_TOKENS = 400   # max tokens the model can generate per answer

# Load the tokenizer and model, then wrap them in a text-generation pipeline
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=False,              # deterministic output
    return_full_text=False,       # only return the newly generated text, not the prompt
    clean_up_tokenization_spaces=False
)
pipe.model.config.max_length = None
llm = HuggingFacePipeline(pipeline=pipe)

# Vector retriever: semantic similarity search over the Chroma vector store
vector_retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})

# BM25 retriever: keyword search rebuilt in memory from the same chunks
bm25_retriever = BM25Retriever.from_documents(doc_chunks)
bm25_retriever.k = TOP_K

# Assemble the full hybrid-retrieval RAG chain
rag_chain = build_rag_chain(vector_retriever, bm25_retriever, tokenizer, llm)

print("Hybrid RAG chain ready!")

# Quick test question
question = "Who does the Travel & Expense Reimbursement Policy apply to?"
result = rag_chain.invoke(question)
print(f"Question: {question}\n")
print("Answer:", result["answer"])
print("\nSources:", result["sources"])


### 5. Generate `agent.py` for MLflow serving

Writes `agent.py`, a self-contained model file that MLflow will package and serve. It defines `RAGPolicyAgent`, an `mlflow.pyfunc.PythonModel` subclass that, at load time, reconnects to the Chroma vector store, rebuilds the BM25 retriever from the pickled chunks, loads the LLM, and builds the same RAG chain using `rag_core.build_rag_chain(...)`. Its `predict` method takes a user question and returns the generated answer along with a JSON-encoded list of sources.

In [0]:
agent_code = """
import pickle
import json
import pandas as pd
import mlflow
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from rag_core import build_rag_chain

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
TOP_K = 6
MAX_NEW_TOKENS = 400


class RAGPolicyAgent(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        # context.artifacts gives the paths to the vector DB and pickled chunks
        # that were bundled with this model when it was logged
        vector_db_path = context.artifacts["vector_db"]
        chunks_path = context.artifacts["doc_chunks"]

        # Reconnect to the persisted Chroma vector store
        embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL, model_kwargs={"device": "cpu"})
        vector_store = Chroma(persist_directory=vector_db_path, embedding_function=embeddings)

        # Rebuild the BM25 keyword index from the pickled chunks
        with open(chunks_path, "rb") as f:
            doc_chunks = pickle.load(f)

        vector_retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K})
        bm25_retriever = BM25Retriever.from_documents(doc_chunks)
        bm25_retriever.k = TOP_K

        # Load the local LLM used to generate answers
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            return_full_text=False,
            clean_up_tokenization_spaces=False
        )
        llm = HuggingFacePipeline(pipeline=pipe)

        # Build the same hybrid-retrieval RAG chain used in the notebook
        self.rag_chain = build_rag_chain(vector_retriever, bm25_retriever, tokenizer, llm)

    def predict(self, context, model_input):
        # Accept a pandas DataFrame, a dict, or a plain string as input
        if isinstance(model_input, pd.DataFrame):
            query = model_input["user_message"].iloc[0]
        elif isinstance(model_input, dict):
            query = model_input.get("user_message", "")
        else:
            query = str(model_input)

        result = self.rag_chain.invoke(query)
        return {"response": result["answer"], "sources": json.dumps(result["sources"])}


# Tell MLflow which object to use as the servable model
mlflow.models.set_model(RAGPolicyAgent())
"""

with open("agent.py", "w") as f:
    f.write(agent_code)

print("agent.py generated successfully.")


### 6. Register the model to Unity Catalog

Logs `agent.py` as an MLflow model, bundling the Chroma vector database, the pickled chunks, and `rag_core.py` alongside it, then registers it to Unity Catalog under `catalog.schema.model_name` so it can be deployed for serving.

In [0]:
import mlflow
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

catalog = "workspace"
schema = "default"
model_name = "policy_rag_agent"
full_model_path = f"{catalog}.{schema}.{model_name}"

# Define the model's input/output schema: a single question string in,
# an answer string and a JSON-encoded sources string out
input_schema = Schema([ColSpec("string", "user_message")])
output_schema = Schema([ColSpec("string", "response"), ColSpec("string", "sources")])
signature = ModelSignature(inputs=input_schema, outputs=output_schema)
input_example = pd.DataFrame([{"user_message": "What is the travel policy?"}])

with mlflow.start_run(run_name="serving_model_registration"):
    model_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        # rag_core.py is a dependency of agent.py, so it's bundled alongside it
        # and placed on sys.path wherever the model is loaded
        code_paths=["rag_core.py"],
        # the vector DB and pickled chunks are packaged as model artifacts
        artifacts={"vector_db": VECTOR_DB_PATH, "doc_chunks": CHUNKS_PATH},
        signature=signature,
        input_example=input_example,
        registered_model_name=full_model_path
    )
    print(f"Model registered to Unity Catalog: {full_model_path}")


### Ask a question

In [0]:
result = rag_chain.invoke("what the rules of parental leave")
print("Answer:", result["answer"])


### Inspect chunk sizes

In [0]:
# Basic stats on how the chunking turned out
lengths = [len(c.page_content) for c in doc_chunks]
print("Shortest chunk:", min(lengths))
print("Longest chunk:", max(lengths))
print("Average chunk length:", sum(lengths) / len(lengths))


In [0]:
# Pull every chunk, its metadata, and its embedding straight out of Chroma
raw = vector_store.get(include=["documents", "metadatas", "embeddings"])

import pandas as pd
df = pd.DataFrame({
    "chunk_id": raw["ids"],
    "text": raw["documents"],
    "source_pdf": [m.get("source") for m in raw["metadatas"]],
    "page": [m.get("page") for m in raw["metadatas"]],
    "section": [m.get("section") or "General" for m in raw["metadatas"]],
})
# Show a short preview of each embedding vector rather than the full 384 numbers
df["embedding_preview"] = [str(v[:5]) + " ..." for v in raw["embeddings"]]

display(df)

In [0]:
# Look at a single chunk in detail, including its raw embedding vector
i = 5
print("Text:", raw["documents"][i][:150])
print("Section:", raw["metadatas"][i].get("section"))
print("Page:", raw["metadatas"][i].get("page"))
print("Its vector (first 8 numbers):", raw["embeddings"][i][:8])
print("Vector length:", len(raw["embeddings"][i]))  # should be 384 for all-MiniLM-L6-v2
